In [0]:
%run ./common_utils

In [0]:
CAT = 'machine_health.prod'
AUDIT = CAT + '.pipeline_run_audit'
declare_widgets(model_version='')
mt = dbutils.widgets.get('machine_type')

try:
    mv = dbutils.widgets.get('model_version').strip()
except Exception:
    mv = ''
if not mv:
    mv = 'fan_v1' if mt.lower() == 'fan' else mt.lower() + '_v1'

with audited() as a:
    maybe_disk_fault()
    wm = spark.sql(f'''select coalesce(max(last_reading_time), timestamp'1970-01-01') m
                       from {CAT}.ingest_watermark where machine_type = '{mt}' ''').first().m
    df = (spark.table(f'{CAT}.sensor_readings_raw')
          .where(F.col('machine_type') == mt)
          .where(F.col('reading_time') > F.lit(wm)))
    if not df.isEmpty():
        out = (df.withColumn('is_healthy',
                             F.when(F.col('vibration_rms_mm_s') > 0.5, 0).otherwise(1).cast('bigint'))
                 .withColumn('model_tag', F.lit(mv))
                 .withColumn('run_id', F.lit(a['job_run_id'])))
        out.write.mode('append').saveAsTable(f'{CAT}.predictions_history')
        (out.write.mode('overwrite').option('replaceWhere', scope(mt))
            .saveAsTable(f'{CAT}.predictions_latest'))
        spark.sql(f'DELETE FROM {CAT}.ingest_watermark WHERE ' + scope(mt))
        (out.groupBy('machine_type').agg(F.max('reading_time').alias('last_reading_time'))
            .write.mode('append').saveAsTable(f'{CAT}.ingest_watermark'))

/home/spark-933aa2c4-50d9-49b8-add6-be/.ipykernel/62/command-7089605854406250-1338218180:94: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time=datetime.utcnow()
/home/spark-933aa2c4-50d9-49b8-add6-be/.ipykernel/62/command-7089605854406250-1338218180:108: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  rec['end_time'] = datetime.utcnow()
